## Vector Embedding With MinSearch
Minsearch is a in-memory search library will help us wrap the logic of embedding the query computed dot products and filter down to the top matches.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [1]:
# Loading data source
from ingest import load_faq_data

documents = load_faq_data()

In [2]:
# Preparing data - embedding both question + answer together
# question and answer will be embedded both together so that a query can match against both the question text and answer text in the index.
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [3]:
# Next step is to generate embeddings
# We need to embed in batches as processing can take a long time
# import tqdm to review process
from tqdm.auto import tqdm

In [6]:
# Now chunk the dataset into batches and encode each batch
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [7]:
# Next we turn it into a 2-D array (matrix) where row = docs and cols = dimension of the vector which represent clues or meanings of the text the model has learned and is represented by a number
import numpy as np
X = np.array(vectors)
X

array([[-0.02670616, -0.12245759,  0.01594421, ..., -0.0023065 ,
        -0.11218402, -0.02365559],
       [-0.01099559, -0.11074748, -0.02536943, ...,  0.09022236,
        -0.02697359,  0.01975664],
       [-0.08896557, -0.06128179,  0.00775611, ...,  0.04059712,
         0.00479282, -0.02745942],
       ...,
       [-0.03652923,  0.0141543 , -0.06838647, ...,  0.04316789,
         0.08105531, -0.02148631],
       [-0.1309159 , -0.06990598, -0.0093188 , ..., -0.00044342,
        -0.01285736,  0.01426917],
       [-0.07984782,  0.01926984,  0.02544974, ..., -0.03368021,
        -0.01884026,  0.05837055]], shape=(1350, 384), dtype=float32)

In [8]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [2]:
vindex

NameError: name 'vindex' is not defined

In [9]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

The above cell computes the dot product between each vector after filtering and our query vector

In [10]:
# Top result
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

## Filtering by course
Like the text index, we can filter by keyword fields. This matters for user experience. A student in LLM Zoom Camp doesn't care about answers from the data engineering course. So we narrow to their course first, then score only within it.

In [11]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [12]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co